## Question 1

In [128]:
import numpy as np

def generer_approbation_profil(n, m, polarisation = 0.0, bruit = 0.05): # n = nombre de votantes, m = nombre de candidates
    
    a = np.random.randint(0, 2, m)
    a_barre = 1 - a
    profil = []

    for _ in range(n):

        if np.random.rand() < (1-polarisation):
            vote = a.copy()
        else:
            if np.random.choice([True, False]):
                vote = a_barre.copy()
            else:
                vote = a.copy()
        
        # ajout du bruit
        for j in range(m):
            if np.random.rand() < bruit:
                vote[j] = 1 - vote[j]
        
        profil.append(vote)

    return np.array(profil)

# Exemple :
#Très peu polarisé :
if __name__ == "__main__":
    p = generer_approbation_profil(n=10, m=5, polarisation=0.0, bruit=0)
    print("Profil très peu polarisé")
    print(p)

#Très polarisé :
if __name__ == "__main__":
    p = generer_approbation_profil(n=10, m=5, polarisation=1.0, bruit=0)
    print("Profil très polarisé")
    print(p)


Profil très peu polarisé
[[1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 0 0 0]
 [1 1 0 0 0]]
Profil très polarisé
[[1 1 1 0 0]
 [0 0 0 1 1]
 [1 1 1 0 0]
 [0 0 0 1 1]
 [1 1 1 0 0]
 [1 1 1 0 0]
 [0 0 0 1 1]
 [0 0 0 1 1]
 [0 0 0 1 1]
 [0 0 0 1 1]]


## Question 2

In [129]:
def generer_ordre_profil(n, m, polarisation = 0.0, bruit = 0.05):

    ordre = np.random.permutation(m)
    ordre_barre = ordre[::-1]
    
    profil = []

    for _ in range(n):

        if np.random.rand() < (1-polarisation):
            vote = ordre.copy()
        else:
            if np.random.choice([True, False]):
                vote = ordre_barre.copy()
            else:
                vote = ordre.copy()

        if np.random.rand() < bruit:
            i, j = np.random.choice(m, 2, replace=False)
            vote[i], vote[j] = vote[j], vote[i]

        profil.append(vote)

    return np.array(profil)

# Exemple :
#Très peu polarisé :
if __name__ == "__main__":
    p = generer_ordre_profil(n=10, m=5, polarisation=0.0, bruit=0)
    print("Profil très peu polarisé")
    print(p)

#Très polarisé :
if __name__ == "__main__":
    p = generer_ordre_profil(n=10, m=5, polarisation=1.0, bruit=0)
    print("Profil très polarisé")
    print(p)


Profil très peu polarisé
[[0 4 1 3 2]
 [0 4 1 3 2]
 [0 4 1 3 2]
 [0 4 1 3 2]
 [0 4 1 3 2]
 [0 4 1 3 2]
 [0 4 1 3 2]
 [0 4 1 3 2]
 [0 4 1 3 2]
 [0 4 1 3 2]]
Profil très polarisé
[[3 0 2 4 1]
 [1 4 2 0 3]
 [3 0 2 4 1]
 [3 0 2 4 1]
 [3 0 2 4 1]
 [1 4 2 0 3]
 [3 0 2 4 1]
 [3 0 2 4 1]
 [1 4 2 0 3]
 [3 0 2 4 1]]


## Question 3

In [130]:
from enum import Enum
import numpy as np

class TypeDeVote(Enum):
    APPROBATION = 0
    ORDRE_TOTAUX = 1

def nbr_votantes_pref(profil, k, l, type_de_vote):
    """Étant donné un profil et deux candidates d'indice k et l, cette méthode 
    retourne de nombre de votantes qui préfèrent ck à cl, étant donné le type de vote.
    """
    match type_de_vote:
        case TypeDeVote.APPROBATION:
            return np.sum((profil[:,k] == 1) & (profil[:,l] == 0))
        case TypeDeVote.ORDRE_TOTAUX:
            n_kl = 0
            for vote in profil:
                position_k = np.where(vote == k)[0][0]
                position_l = np.where(vote == l)[0][0]
                if position_k < position_l:
                    n_kl += 1
            return n_kl


def diff_absolue(profil, k, l, type_de_vote):
    n_kl = nbr_votantes_pref(profil, k, l, type_de_vote)
    n_lk = nbr_votantes_pref(profil, l, k, type_de_vote)
    return abs(n_kl - n_lk)


def ensemble_des_diff_absolue(profil, type_de_vote):
    n, m = profil.shape
    d_valeurs = []
    for k in range(m):
        for l in range(k+1, m):
            d = diff_absolue(profil, k, l, type_de_vote)
            d_valeurs.append(d)
    return np.array(d_valeurs)


def ensemble_des_diff_absolue_approbation(profil):
    return ensemble_des_diff_absolue(profil, TypeDeVote.APPROBATION)


def ensemble_des_diff_absolue_ordre_totaux(profil):
        return ensemble_des_diff_absolue(profil, TypeDeVote.ORDRE_TOTAUX)


#Exemple : 
if __name__ == "__main__":
    p = generer_approbation_profil(n=10, m=5, polarisation=1.0)
    print(ensemble_des_diff_absolue_approbation(p))

        
#Exemple : 
if __name__ == "__main__":
    p = generer_ordre_profil(n=10, m=5, polarisation=1.0)
    print(ensemble_des_diff_absolue_ordre_totaux(p))
            


[1 2 1 1 1 0 0 1 1 0]
[6 6 6 6 6 6 6 6 6 6]


## Ce que chat m'a fait en attendant de remplacer par les réponses de Lucie

In [1]:
import numpy as np
import random

def generer_profil_approbations(n, m, niveau_polarisation):
    """
    Génère un profil de votes par approbations avec un certain niveau de polarisation.

    Paramètres:
    - n (int): nombre de votantes
    - m (int): nombre de candidates
    - niveau_polarisation (float): niveau de polarisation (0 = faible, 1 = forte)

    Retourne:
    - profil (list): profil de votes par approbations
    """
    # Génération d'un bulletin aléatoire
    bulletin_aleatoire = np.random.randint(0, 2, size=m)

    # Génération du profil
    profil = []

    # Nombre de votantes dans chaque cluster
    n_cluster_1 = int(n * (1 - niveau_polarisation) / 2)
    n_cluster_2 = n - n_cluster_1

    # Cluster 1: bulletins proches de bulletin_aleatoire
    for _ in range(n_cluster_1):
        bulletin = bulletin_aleatoire.copy()
        # Ajout de bruit pour éviter une polarisation parfaite
        for i in range(m):
            if random.random() < 0.1:  # 10% de chance de changer un vote
                bulletin[i] = 1 - bulletin[i]
        profil.append(bulletin)

    # Cluster 2: bulletins proches de l'opposé de bulletin_aleatoire
    bulletin_oppose = 1 - bulletin_aleatoire
    for _ in range(n_cluster_2):
        bulletin = bulletin_oppose.copy()
        # Ajout de bruit pour éviter une polarisation parfaite
        for i in range(m):
            if random.random() < 0.1:  # 10% de chance de changer un vote
                bulletin[i] = 1 - bulletin[i]
        profil.append(bulletin)

    # Mélange du profil pour éviter un ordre déterministe
    random.shuffle(profil)

    return profil

def generer_profil_ordres_totaux(n, m, niveau_polarisation):
    """
    Génère un profil de votes par ordres totaux avec un certain niveau de polarisation.

    Paramètres:
    - n (int): nombre de votantes
    - m (int): nombre de candidates
    - niveau_polarisation (float): niveau de polarisation (0 = faible, 1 = forte)

    Retourne:
    - profil (list): profil de votes par ordres totaux
    """
    # Génération d'un ordre total aléatoire
    ordre_aleatoire = list(range(m))
    random.shuffle(ordre_aleatoire)

    # Génération du profil
    profil = []

    # Nombre de votantes dans chaque cluster
    n_cluster_1 = int(n * (1 - niveau_polarisation) / 2)
    n_cluster_2 = n - n_cluster_1

    # Cluster 1: ordres proches de ordre_aleatoire
    for _ in range(n_cluster_1):
        ordre = ordre_aleatoire.copy()
        # Ajout de bruit pour éviter une polarisation parfaite
        if random.random() < 0.3:  # 30% de chance de permuter deux éléments
            i, j = random.sample(range(m), 2)
            ordre[i], ordre[j] = ordre[j], ordre[i]
        profil.append(ordre)

    # Cluster 2: ordres proches de l'opposé de ordre_aleatoire
    ordre_oppose = ordre_aleatoire[::-1]
    for _ in range(n_cluster_2):
        ordre = ordre_oppose.copy()
        # Ajout de bruit pour éviter une polarisation parfaite
        if random.random() < 0.3:  # 30% de chance de permuter deux éléments
            i, j = random.sample(range(m), 2)
            ordre[i], ordre[j] = ordre[j], ordre[i]
        profil.append(ordre)

    # Mélange du profil pour éviter un ordre déterministe
    random.shuffle(profil)

    return profil

# Exemple d'utilisation
n = 10  # nombre de votantes
m = 4   # nombre de candidates

# Génération de profils avec différents niveaux de polarisation
profil_approbations_faible = generer_profil_approbations(n, m, niveau_polarisation=0.1)
profil_approbations_fort = generer_profil_approbations(n, m, niveau_polarisation=0.9)

profil_ordres_faible = generer_profil_ordres_totaux(n, m, niveau_polarisation=0.1)
profil_ordres_fort = generer_profil_ordres_totaux(n, m, niveau_polarisation=0.9)

print("Profil d'approbations (faible polarisation) :")
print(profil_approbations_faible)
print("\nProfil d'approbations (forte polarisation) :")
print(profil_approbations_fort)

print("\nProfil d'ordres totaux (faible polarisation) :")
print(profil_ordres_faible)
print("\nProfil d'ordres totaux (forte polarisation) :")
print(profil_ordres_fort)


Profil d'approbations (faible polarisation) :
[array([0, 0, 1, 1]), array([1, 1, 0, 0]), array([0, 0, 1, 1]), array([0, 1, 1, 0]), array([0, 1, 0, 0]), array([0, 1, 0, 0]), array([0, 0, 1, 1]), array([1, 0, 0, 1]), array([0, 0, 1, 1]), array([0, 1, 0, 0])]

Profil d'approbations (forte polarisation) :
[array([0, 1, 1, 1]), array([0, 1, 1, 1]), array([0, 1, 1, 1]), array([0, 1, 1, 1]), array([0, 1, 0, 1]), array([0, 1, 0, 0]), array([0, 1, 1, 1]), array([0, 1, 1, 1]), array([0, 0, 1, 1]), array([0, 0, 1, 1])]

Profil d'ordres totaux (faible polarisation) :
[[2, 3, 0, 1], [2, 3, 0, 1], [1, 0, 3, 2], [2, 3, 0, 1], [1, 0, 3, 2], [2, 3, 0, 1], [2, 3, 0, 1], [1, 0, 3, 2], [1, 3, 0, 2], [1, 3, 0, 2]]

Profil d'ordres totaux (forte polarisation) :
[[0, 1, 3, 2], [0, 1, 3, 2], [0, 1, 3, 2], [0, 1, 3, 2], [0, 1, 3, 2], [0, 1, 3, 2], [0, 1, 3, 2], [0, 1, 3, 2], [1, 0, 3, 2], [0, 1, 3, 2]]


## Question 8 : 

In [ ]:
def distance_hamming(a, b):
    # a ^ b est le XOR bit à bit de a et b: contient des 1 aux positions où a et b diffèrent, et des 0 aux positions où a et b sont identiques.
    np.sum(a ^ b)

In [ ]:
def distance_spearman(a, b):
    return np.sum(abs(np.subtract(a, b)))

2


## Question 12

In [183]:
import random

def calcul_consensus_approbation(profil):
    n,m = profil.shape
    consensus = [0]*m
    for i in range (m) :
        somme = 0 
        for votes in profil : 
            somme = somme + votes[i]

        if somme > n/2 : 
            consensus[i] = 1
        if somme < n/2 : 
            consensus[i] = 0  
        else :
            consensus[i] = random.randint(0,1)  
    return consensus
    
def calcul_vote_approbation(profil):
    consensus = calcul_consensus_approbation(profil)
    u1 = 0
    for votes in profil :
        u1 += distance_hamming(votes, consensus)
    return u1

print(calcul_vote_approbation(generer_approbation_profil(n=5, m=10, polarisation=0.3)))

5


In [198]:
from scipy.optimize import linear_sum_assignment

def calcul_consensus_vote_ordre_totaux(profil):
    n,m = profil.shape
    consensus=[0]*m
    rank = np.arange(m)
    candidates = np.arange(m)
    u1 = 0
    poids = np.zeros((m,m))
    for votes in profil :
        for position in range(m):
            for candidate in range (m) :
                poids[position, candidate] += abs(votes[candidate] - position)
    rank_ind, candidate_ind = linear_sum_assignment(poids)

    for i in range(m):
        consensus[rank_ind[i]] = int(candidate_ind[i])

    for i in range(m):
        u1 += int(poids[rank_ind[i], candidate_ind[i]])
    return [consensus, u1]

print(calcul_consensus_vote_ordre_totaux(generer_ordre_profil(n=5, m=10, polarisation=0.7)))

[[4, 9, 5, 2, 1, 6, 0, 8, 3, 7], 76]


## Question 13

In [195]:
import random

def calcul_centroide_approbation(profil, max_iter):
    n,m = profil.shape
    a1 = np.random.randint(0,2, size=m) 
    a2 = np.random.randint(0,2, size = m) 

    for i in range (max_iter) :
        cluster1 = []
        cluster2 = []

        for votes in profil : 
            if distance_hamming(votes, a1) < distance_hamming(votes, a2) : 
                cluster1.append(votes)
            else : 
                cluster2.append(votes)

        if len(cluster1) > 0:
            a1BIS = calcul_consensus_approbation(np.array(cluster1))
        else:
            a1BIS = a1  # Garder l'ancien centroïde si le cluster est vide

        if len(cluster2) > 0:
            a2BIS = calcul_consensus_approbation(np.array(cluster2))
        else:
            a2BIS = a2  # Garder l'ancien centroïde si le cluster est vide


        if np.array_equal(a1, a1BIS) and np.array_equal(a2, a2BIS):
            u2 = 0 
            for bulletin in cluster1:
                u2 += distance_hamming(bulletin, a1)
            for bulletin in cluster2:
                u2 += distance_hamming(bulletin, a2)
            return u2

        else : 
            a1 = a1BIS 
            a2 = a2BIS
            
    return ValueError

profil = generer_approbation_profil(n=5, m=10, polarisation=0.0)
print(calcul_centroide_approbation(profil, max_iter=100))

8


In [200]:
import random

def calcul_centroide_ordre_totaux(profil, max_iter):
    n,m = profil.shape
    a1 = np.random.randint(0,2, size=m) 
    a2 = np.random.randint(0,2, size = m) 

    for i in range (max_iter) :
        cluster1 = []
        cluster2 = []

        for votes in profil : 
            if distance_spearman(votes, a1) < distance_spearman(votes, a2) : 
                cluster1.append(votes)
            else : 
                cluster2.append(votes)
        if len(cluster1) > 0:
            a1BIS = calcul_consensus_vote_ordre_totaux(profil)[0](np.array(cluster1))
        else:
            a1BIS = a1  # Garder l'ancien centroïde si le cluster est vide

        if len(cluster2) > 0:
            a2BIS = calcul_consensus_vote_ordre_totaux(profil)[0](np.array(cluster2))
        else:
            a2BIS = a2  # Garder l'ancien centroïde si le cluster est vide


        if np.array_equal(a1, a1BIS) and np.array_equal(a2, a2BIS):
            u2 = 0 
            for bulletin in cluster1:
                u2 += distance_spearman(bulletin, a1)
            for bulletin in cluster2:
                u2 += distance_spearman(bulletin, a2)
            return u2

        else : 
            a1 = a1BIS 
            a2 = a2BIS
            
    return ValueError

profil = generer_approbation_profil(n=5, m=10, polarisation=0.0)
print(calcul_centroide_ordre_totaux(profil, max_iter=100))

TypeError: 'list' object is not callable

## Question 13 de python

In [ ]:
import random

def distance_hamming(a, b):
    """Calcule la distance de Hamming entre deux bulletins."""
    return sum(1 for i in range(len(a)) if a[i] != b[i])

def calcul_centroide_consensus(cluster):
    if not cluster:
        return None
    
    cluster = np.array(cluster)
    n, m = cluster.shape
    centroide = np.zeros(m, dtype=int)

    for i in range(m):
        somme = np.sum(cluster[:, i])
        if somme > n / 2:
            centroide[i] = 1
        elif somme < n / 2:
            centroide[i] = 0
        else:
            centroide[i] = np.random.randint(0, 2)

    return centroide

def calcul_u2_approbation(profil, max_iter=100):
    """Estime u_2*(p) pour les votes par approbation avec k-means."""
    n = len(profil)
    if n == 0:
        return 0
    m = len(profil[0])

    # Initialisation aléatoire des centroïdes
    centroide1 = random.choice(profil)
    centroide2 = [1 - x for x in centroide1]

    for _ in range(max_iter):
        # Étape 1 : Affectation des bulletins aux clusters
        cluster1 = []
        cluster2 = []
        for bulletin in profil:
            dist1 = distance_hamming(bulletin, centroide1)
            dist2 = distance_hamming(bulletin, centroide2)
            if dist1 <= dist2:
                cluster1.append(bulletin)
            else:
                cluster2.append(bulletin)

        # Étape 2 : Mise à jour des centroïdes en utilisant la logique de la question 12
        new_centroide1 = calcul_centroide_consensus(cluster1)
        new_centroide2 = calcul_centroide_consensus(cluster2)

        if new_centroide1 is None:
            new_centroide1 = centroide1
        if new_centroide2 is None:
            new_centroide2 = centroide2

        # Vérification de la convergence
        if np.array_equal(new_centroide1, centroide1) and np.array_equal(new_centroide2, centroide2):
            break

        centroide1, centroide2 = new_centroide1, new_centroide2

    # Calcul de u_2*(p)
    u2 = 0
    for bulletin in cluster1:
        u2 += distance_hamming(bulletin, centroide1)
    for bulletin in cluster2:
        u2 += distance_hamming(bulletin, centroide2)

    return u2

# Exemple d'utilisation
n = 10
m = 4
profil_approbation = generer_profil_approbations(n, m, 0.5)
u2_approbation = calcul_u2_approbation(profil_approbation)
print(f"Estimation de u_2*(p) pour les votes par approbation : {u2_approbation}")


Estimation de u_2*(p) pour les votes par approbation : 2


In [67]:
from scipy.optimize import linear_sum_assignment
import numpy as np

def distance_spearman(ordre1, ordre2):
    """Calcule la distance de Spearman entre deux ordres totaux."""
    return sum(abs(ordre1.index(c) - ordre2.index(c)) for c in range(len(ordre1)))

def calcul_centroide_consensus_ordres(cluster):
    """Calcule l'ordre consensus pour un cluster donné en utilisant la logique de la question 12."""
    if not cluster:
        return None
    m = len(cluster[0])
    poids = np.zeros((m, m))

    for ordre in cluster:
        for position in range(m):
            for candidate in range(m):
                poids[position][candidate] += abs(ordre.index(candidate) - position)

    row_ind, col_ind = linear_sum_assignment(poids)
    ordre_consensus = [0] * m
    for pos, cand in zip(row_ind, col_ind):
        ordre_consensus[pos] = cand
    return ordre_consensus

def calcul_u2_ordres_totaux(profil, max_iter=100):
    """Estime u_2*(p) pour les votes par ordres totaux avec k-means."""
    n = len(profil)
    if n == 0:
        return 0

    # Initialisation aléatoire des centroïdes
    centroide1 = random.choice(profil)
    centroide2 = centroide1[::-1]

    for _ in range(max_iter):
        # Étape 1 : Affectation des ordres aux clusters
        cluster1 = []
        cluster2 = []
        for ordre in profil:
            dist1 = distance_spearman(ordre, centroide1)
            dist2 = distance_spearman(ordre, centroide2)
            if dist1 <= dist2:
                cluster1.append(ordre)
            else:
                cluster2.append(ordre)

        # Étape 2 : Mise à jour des centroïdes en utilisant la logique de la question 12
        new_centroide1 = calcul_centroide_consensus_ordres(cluster1)
        new_centroide2 = calcul_centroide_consensus_ordres(cluster2)

        if new_centroide1 is None:
            new_centroide1 = centroide1
        if new_centroide2 is None:
            new_centroide2 = centroide2

        # Vérification de la convergence
        if (new_centroide1 == centroide1) and (new_centroide2 == centroide2):
            break

        centroide1, centroide2 = new_centroide1, new_centroide2
    
    # Calcul de u_2*(p)
    u2 = 0
    for ordre in cluster1:
        u2 += distance_spearman(ordre, centroide1)
    for ordre in cluster2:
        u2 += distance_spearman(ordre, centroide2)

    return u2

# Exemple d'utilisation
profil_ordres = generer_profil_ordres_totaux(n, m, 0.5)
u2_ordres = calcul_u2_ordres_totaux(profil_ordres)
print(f"Estimation de u_2*(p) pour les votes par ordres totaux : {u2_ordres}")


Estimation de u_2*(p) pour les votes par ordres totaux : 8


## Question 14 : 

In [8]:
def phi_hamming(p):
    #Ajout de ces 4 lignes après avoir constaté à la question 15 que sans cela on pouvait avoir des division par 0
    if n == 0 : 
        return ValueError
    if m == 0 :
        return ValueError
    
    u1_h = calcul_vote_approbation(p)
    u2_h = calcul_u2_approbation(p)
    numerateur = 2 * (u1_h-u2_h)
    denominateur = n*m
    calcul = numerateur/denominateur
    return calcul
    

In [9]:
def phi_spearman(p):
    #Ajout de ces 4 lignes après avoir constaté à la question 15 que sans cela on pouvait avoir des division par 0
    if n == 0 : 
        return ValueError
    if m == 0 :
        return ValueError  
    u1_s = calcul_vote_totaux(p)
    u2_s = calcul_u2_ordres_totaux(p)
    num = 4*(u1_s-u2_s)
    den = n * (m**2)
    res = num/den
    return res

### Test

Ici, on a une polarisation de 0.9 soit proche de 1. On devrait donc obtenir un 𝛟 proche de 1.

In [10]:
p = generer_profil_approbations(20, 5, 0.9)
print(phi_hamming(p))

IndexError: list index out of range

Ici, on a une polarisation très faible (de 0.1). On devrait donc obtenir un 𝛟 proche de 0.

In [11]:
p = generer_profil_approbations(20, 5, 0.1)
print(phi_hamming(p))

IndexError: list index out of range

## Question 15 : 

### Évolution des mesures 𝜑_dh

In [12]:
import matplotlib.pyplot as plt
niveau_polarisation = [k/10 for k in range (11)]

phi = [] #on créer une liste pour pouvoir enregistré les différentes valeurs données

for i in niveau_polarisation :
    moyenne = 0
    nb_tests = 20
    for n in range (nb_tests):
        p = generer_profil_approbations(20,5,i)
        moyenne = moyenne + phi_hamming(p)
    phi.append(moyenne/nb_tests)

plt.plot(niveau_polarisation,phi)
plt.xlabel("Paramètres de polarisation p")
plt.ylabel("phi_dh(p)")
plt.title("Évolution de la polarisation en fonction de la distance de Hamming")
plt.show()



TypeError: unsupported operand type(s) for +: 'int' and 'type'

### Évolution des mesures 𝜑_ds

In [ ]:
niveau_polarisation = [k/10 for k in range (11)]

phi = [] #on créer une liste pour pouvoir enregistré les différentes valeurs données

for i in niveau_polarisation :
    moyenne = 0
    nb_tests = 20
    for n in range (nb_tests):
        p = generer_profil_ordres_totaux(20,5,i)
        moyenne = moyenne + phi_spearman(p)
    phi.append(moyenne/nb_tests)

plt.plot(niveau_polarisation,phi)
plt.xlabel("Paramètres de polarisation p")
plt.ylabel("phi_ds(p)")
plt.title("Évolution de la polarisation en fonction de la distance de Spearman")
plt.show()


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'type'